# JOINs: INNER, LEFT, Múltiplos e Self-Join
---

## Contexto: o relatório de clientes está errado!

O relatório mensal mostrou que a empresa tem 120 clientes, mas o gerente sabe que são 150.  
Você investiga: o código usa `JOIN` onde deveria usar `LEFT JOIN`, **excluindo clientes sem pedidos**.  
Esse é um bug silencioso clássico em JOINs e você vai aprender a evitá-lo.

Nesta aula você vai aprender:
- A diferença prática entre INNER JOIN e LEFT OUTER JOIN
- JOINs múltiplos encadeados
- Self-join: comparar a tabela com ela mesma
- JOIN + GROUP BY para métricas
- Como inspecionar o SQL gerado para diagnosticar erros

---

## 1. Configuração

In [ ]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect, 
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists, any_
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload, aliased
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

In [ ]:
# Verificar tabelas com SQLAlchemy
insp = inspect(engine)

print(insp.get_table_names())

## 2. JOIN via relacionamento vs JOIN explícito

O SQLAlchemy oferece duas formas de escrever um JOIN:

```python
# Via relacionamento ORM — o SQLAlchemy sabe o ON automaticamente
select(Cliente).join(Cliente.pedidos)

# Explícito — você escreve o ON manualmente
select(Cliente).join(Pedido, Cliente.id == Pedido.cliente_id)
```

**Use relacionamento** quando ele existe e o JOIN é direto.  
**Use explícito** quando precisar de condições extras no ON.

## 3. INNER JOIN — só quem tem par nos dois lados

O `INNER JOIN` funciona como a interseção de dois conjuntos. Ele percorre a tabela de `Clientes` e a tabela de `Pedidos` e só traz os resultados onde existe uma correspondência exata entre as duas, baseada na chave estrangeira.

In [ ]:
# código:

## 4. LEFT OUTER JOIN — todos da esquerda, com ou sem par

`.outerjoin()` retorna **todos os clientes**, mesmo os sem pedido.  
Onde não há par, as colunas do lado direito aparecem como `None`.

In [ ]:
# código:

## 5. JOINs múltiplos — encadeando tabelas

Em análises de vendas, a cadeia completa é:  
**Cliente → Pedido → ItemPedido → Produto**

In [ ]:
# código:

## 6. Self-join — comparando a tabela com ela mesma

Às vezes a pergunta envolve comparar linhas da **mesma tabela**.  
Exemplo: *"Quais pares de clientes moram na mesma cidade?"*

Para isso, usamos `aliased()` — como ter duas cópias da mesma tabela com nomes diferentes.

In [ ]:
# códigos:

## 7. JOIN + GROUP BY — calculando métricas por grupo

In [ ]:
# código

# 8. Visualizando os SQL gerados pelo ORM

Uma das melhores formas de aprender SQLAlchemy é ver o SQL real que ele está enviando para o banco de dados.

In [ ]:
# código

---

## Exercício: Usando IA para isso

**Prompt para otimizar JOINs múltiplos:**
```
Tenho esta query SQLAlchemy com 3 JOINs encadeados que está lenta.
Sugira: índices que deveriam existir, reescrita mais eficiente,
ou se subconsulta seria mais adequada em alguma parte.
```

```python 
with Session(engine) as session:

    stmt = (
        select(            
            Cliente.nome.label("cliente"),          
            Pedido.id.label("pedido"),
            Produto.nome.label("produto"),
            Produto.categoria,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,            
            (ItemPedido.quantidade * ItemPedido.preco_venda).label("subtotal")
        )
        
        .join(Pedido, Cliente.id == Pedido.cliente_id)        
        .join(ItemPedido, Pedido.id == ItemPedido.pedido_id)        
        .join(Produto, ItemPedido.produto_id == Produto.id)        
        .order_by("subtotal")
    )

    linhas = session.execute(stmt).all()    
    print(f"Itens vendidos (com contexto completo): {len(linhas)}")
    print("-" * 80)

    # Percorre apenas os 3 primeiros resultados
    for l in linhas[:3]:
        
        # Exibe as informações formatadas de cada item
        print(f"  {l.cliente} → Pedido #{l.pedido} | {l.quantidade}x {l.produto} | R${l.subtotal}")
```


---

### Resposta:Código gerado pelo ChatGPT

In [ ]:
with Session(engine) as session:

    subtotal = (ItemPedido.quantidade * ItemPedido.preco_venda)

    stmt = (
        select(            
            Cliente.nome.label("cliente"),          
            Pedido.id.label("pedido"),
            Produto.nome.label("produto"),
            Produto.categoria,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,            
            subtotal.label("subtotal")
        )
        .join(Pedido, Cliente.id == Pedido.cliente_id)
        .join(ItemPedido, Pedido.id == ItemPedido.pedido_id)
        .join(Produto, ItemPedido.produto_id == Produto.id)
        .order_by(subtotal.desc())
        .limit(3)
    )

    linhas = session.execute(stmt).all()
    print(f"Itens vendidos (com contexto completo): {len(linhas)}")
    print("-" * 80)

    # Percorre apenas os 3 primeiros resultados
    for l in linhas[:3]:
        
        # Exibe as informações formatadas de cada item
        print(f"  {l.cliente} → Pedido #{l.pedido} | {l.quantidade}x {l.produto} | R${l.subtotal}")